# V8.1 Training - Direct GitHub URL Load

No Google Drive sync issues - load directly from GitHub raw files.

In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy requests

In [ ]:
import torch, pandas as pd, numpy as np, json, io, requests
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings; warnings.filterwarnings('ignore')

MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# Load from GitHub RAW (no Google Drive path issues!)
GITHUB_BASE = "https://raw.githubusercontent.com/Das-rebel/autonomous_laughter_prediction_essential/main/data/v8_1_final"

def load_jsonl_from_url(url):
    resp = requests.get(url)
    resp.raise_for_status()
    lines = resp.text.strip().split('\n')
    return pd.DataFrame([json.loads(l) for l in lines])

print("Loading V8.1 from GitHub...")
train_df = load_jsonl_from_url(f'{GITHUB_BASE}/train.jsonl')
val_df = load_jsonl_from_url(f'{GITHUB_BASE}/valid.jsonl')
print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Languages: {train_df['language'].value_counts().to_dict()}")

In [ ]:
# Prepare labels
if 'labels' in train_df.columns and isinstance(train_df.iloc[0].get('labels', []), list):
    train_df['ex_label'] = train_df['labels'].apply(lambda x: 1 if any(v == 1 for v in x) else 0)
    val_df['ex_label'] = val_df['labels'].apply(lambda x: 1 if any(v == 1 for v in x) else 0)
elif 'label' in train_df.columns:
    train_df['ex_label'] = train_df['label']
    val_df['ex_label'] = val_df['label']
print(f"Label dist: {train_df['ex_label'].value_counts().to_dict()}")

In [ ]:
class V8DS(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        txt = ' '.join(self.texts[idx]) if isinstance(self.texts[idx], list) else str(self.texts[idx])
        enc = self.tok(txt, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'label': torch.tensor(self.labels[idx], dtype=torch.long)}

class XLMR(torch.nn.Module):
    def __init__(self, mname):
        super().__init__()
        self.enc = AutoModel.from_pretrained(mname)
        self.drop = torch.nn.Dropout(0.2)
        self.cls = torch.nn.Linear(self.enc.config.hidden_size, 2)
    def forward(self, ids, mask):
        return self.cls(self.drop(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]))

In [ ]:
# Prepare data
train_texts = train_df['words'].apply(lambda x: ' '.join(x)).tolist()
train_labels = train_df['ex_label'].tolist()
val_texts = val_df['words'].apply(lambda x: ' '.join(x)).tolist()
val_labels = val_df['ex_label'].tolist()
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = V8DS(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = V8DS(val_texts, val_labels, tokenizer, MAX_LEN)
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f"Batches: train={len(train_ld)}, val={len(val_ld)}")

In [ ]:
model = XLMR(MODEL_NAME).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(train_ld)*EPOCHS), num_training_steps=len(train_ld)*EPOCHS)
print("Model ready!")

In [ ]:
def train_ep(m, ld, opt, sch):
    m.train()
    tot, preds, labs = 0, [], []
    for b in ld:
        opt.zero_grad()
        logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
        loss = torch.nn.CrossEntropyLoss()(logits, b['label'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step(); sch.step()
        tot += loss.item()
        preds.extend(torch.argmax(logits, 1).cpu().numpy())
        labs.extend(b['label'].numpy())
    return tot/len(ld), f1_score(labs, preds, average='weighted')

def eval_ep(m, ld):
    m.eval()
    tot, preds, labs = 0, [], []
    with torch.no_grad():
        for b in ld:
            logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
            loss = torch.nn.CrossEntropyLoss()(logits, b['label'].to(DEVICE))
            tot += loss.item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labs.extend(b['label'].numpy())
    return tot/len(ld), f1_score(labs, preds, average='weighted'), preds, labs

In [ ]:
best_f1 = 0
hist = {'tl': [], 'tf': [], 'vl': [], 'vf': []}
print("Training V8.1...")
for ep in range(EPOCHS):
    tl, tf = train_ep(model, train_ld, opt, sched)
    vl, vf, _, _ = eval_ep(model, val_ld)
    hist['tl'].append(tl); hist['tf'].append(tf); hist['vl'].append(vl); hist['vf'].append(vf)
    print(f"Ep{ep+1}/{EPOCHS}: TL={tl:.4f} TF={tf:.4f} VL={vl:.4f} VF={vf:.4f}")
    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), '/tmp/best_v8.pt')
        print(f"  -> Saved F1={best_f1:.4f}")
print(f"Best Val F1: {best_f1:.4f}")

In [ ]:
model.load_state_dict(torch.load('/tmp/best_v8.pt'))
_, _, preds, actual = eval_ep(model, val_ld)
val_df['pred'] = preds
print("="*50)
print("RESULTS")
print("="*50)
results = {}
for lang in sorted(val_df['language'].unique()):
    ld = val_df[val_df['language'] == lang]
    f1 = f1_score(ld['ex_label'], ld['pred'], average='weighted')
    results[lang] = f1
    st = "PASS" if f1 >= 0.75 else "FAIL"
    print(f"{lang.upper()}: F1={f1:.4f} [{st}]")
overall = f1_score(actual, preds, average='weighted')
st = "PASS" if overall >= 0.75 else "FAIL"
print(f"OVERALL: F1={overall:.4f} [{st}]")

In [ ]:
# Save to Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('/tmp/best_v8.pt', '/content/drive/MyDrive/v8_1_models/best_v8.pt')
model.save_pretrained('/content/drive/MyDrive/v8_1_models/xlmr_v8')
tokenizer.save_pretrained('/content/drive/MyDrive/v8_1_models/xlmr_v8')
with open('/content/drive/MyDrive/v8_1_models/results.json', 'w') as f:
    json.dump({'overall': overall, 'per_lang': results, 'best': best_f1}, f, indent=2)
print("Saved to Google Drive!")